## 1. Setup Snowflake connection and load data

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "../00_cohort_config"

In [0]:

# VALIDATION: read from validation outputs (not production)
# NOTE: lab data now reads notebook 01's PROCESSED output (forward-filled, cohort-month grain)
# The original read the raw input (kerendia_nba_3_0_lab_query_output) which was at daily grain
df_lab_pat_journey = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_lab_journey_validation")

df_dx_pat_journey = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_dx_journey_validation")

df_rx_pat_journey = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_rx_journey_validation")

df_t2d_patient_pat_journey = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_t2d_patient_validation")

df_px_pat_journey = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_px_validation")

In [0]:
print('df_lab_pat_journey',df_lab_pat_journey.count(), len(df_lab_pat_journey.columns))
print('df_dx_pat_journey',df_dx_pat_journey.count(), len(df_dx_pat_journey.columns))
print('df_rx_pat_journey',df_rx_pat_journey.count(), len(df_rx_pat_journey.columns))
print('df_t2d_patient_pat_journey',df_t2d_patient_pat_journey.count(), len(df_t2d_patient_pat_journey.columns))
print('df_px_pat_journey',df_px_pat_journey.count(), len(df_px_pat_journey.columns))


## 2. Combine Patinet Journey Table

In [0]:
df_spine_pat_journey = (
  df_lab_pat_journey
  .join(df_dx_pat_journey, how = "full", on = ["PATIENT_ID", "DATE"])
  .join(df_rx_pat_journey, how = "full", on = ["PATIENT_ID", "DATE"])
  .orderBy("PATIENT_ID", "DATE")
  .withColumn("DX_STAGE_MAX", col("DX_STAGE_UPDATED"))

  .withColumn("GFR_DX_STAGE", F.coalesce(col("DX_STAGE_MAX"), col("GFR_STAGE")))
 
  .withColumn("GFR_UACR_DX_STAGE", F.coalesce(col("DX_STAGE_MAX"), col("GFR_STAGE"), col("UACR_STAGE")))
)

df_spine_pat_journey.display()

In [0]:
df_spine_pat_journey_with_flag1 = (
  df_spine_pat_journey
  .join(df_t2d_patient_pat_journey, how = "left", on = "PATIENT_ID")
  .na.fill(value=0, subset=["DIABETIC_FLAG"])
)

In [0]:
df_spine_pat_journey_with_flag2 = (
  df_spine_pat_journey_with_flag1
  .join(df_px_pat_journey, how = "left", on = "PATIENT_ID")
  # Compare at month grain: truncate FIRST_DIALYSIS_DATE to month so it aligns with DATE
  # (DATE is now 1st-of-month cohort reference; without this, a mid-month dialysis would lag by 1 month)
  .withColumn("DIALYSIS_FLAG", 
              F.when(col("FIRST_DIALYSIS_DATE") < '2017-01-01', 1).otherwise(
              F.when(F.trunc(F.to_date(col("FIRST_DIALYSIS_DATE")), "month") <= col("DATE"), 1).otherwise(0)))
  .na.fill(value=0, subset=["DIALYSIS_FLAG"])
  .drop("FIRST_DIALYSIS_DATE")
)

In [0]:
df_spine_pat_journey_with_flag2.filter(col("PATIENT_ID") == 29370266).display()


#### NPI_NUMBER, MONTH Level Patient Journey

In [0]:
df_spine_pat_journey_with_flag2.select("TREATMENT_CATEGORY").limit(5).show(truncate=False)

In [0]:
pat_new_and_total_treatments =  (
  df_spine_pat_journey_with_flag2
  # DATE is already at cohort-month grain from upstream notebooks — no need to date_trunc
  .withColumn("SVC_DT_M", col("DATE"))
  .withColumn("TOTAL_TREATMENTS", F.greatest(F.size(col("TREATMENT_CATEGORY")), F.lit(0)))
  .withColumn("TOTAL_TREATMENTS_LAST", F.lag("TOTAL_TREATMENTS", 1).over(Window.partitionBy("PATIENT_ID").orderBy("DATE")))
  .withColumn("TOTAL_NEW_TREATMENTS", F.greatest(col("TOTAL_TREATMENTS") - col("TOTAL_TREATMENTS_LAST"), F.lit(0)))
  .groupBy("PATIENT_ID", "SVC_DT_M")
  .agg(F.max("TOTAL_TREATMENTS").alias("TOTAL_TREATMENTS"),
       F.sum("TOTAL_NEW_TREATMENTS").alias("TOTAL_NEW_TREATMENTS")
  )
)

pat_new_and_total_treatments = pat_new_and_total_treatments.withColumn("PULL_DT", F.current_date())

pat_new_and_total_treatments.filter(col("PATIENT_ID") == 29370266).display()

In [0]:
pat_new_ckd_stage =  (
  df_spine_pat_journey_with_flag2
  # DATE is already at cohort-month grain from upstream notebooks
  .withColumn("SVC_DT_M", col("DATE"))
  .withColumn("CKD_STAGE", col("GFR_DX_STAGE"))
  .groupBy("PATIENT_ID", "SVC_DT_M")
  .agg(F.max("CKD_STAGE").alias("CKD_STAGE"))
  .withColumn("CKD_STAGE_LAST", F.lag("CKD_STAGE", 1).over(Window.partitionBy("PATIENT_ID").orderBy("SVC_DT_M")))
  .withColumn("NEW_CKD_STAGE_FLAG", F.when(col("CKD_STAGE") == col("CKD_STAGE_LAST"), 0).otherwise(1))
  .withColumn("NEW_CKD_STAGE", F.when(col("NEW_CKD_STAGE_FLAG") == 1, col("CKD_STAGE")).otherwise(None))
  .select("PATIENT_ID", "SVC_DT_M", "NEW_CKD_STAGE")
)

pat_new_ckd_stage = pat_new_ckd_stage.withColumn("PULL_DT", F.current_date())

pat_new_ckd_stage.filter(col("PATIENT_ID") == 29370266).display()

In [0]:
pat_gfr_dx_stage_and_treatment_category = (
  df_spine_pat_journey_with_flag2
  # DATE is already at cohort-month grain from upstream notebooks
  .withColumn("SVC_DT_M", col("DATE"))
  .withColumn("TREATMENT_CATEGORY", F.explode_outer(col("TREATMENT_CATEGORY")))
  .withColumn("CATEGORY", F.explode_outer(col("CATEGORY")))
  .groupBy("PATIENT_ID", "SVC_DT_M")
  .agg(F.max("GFR_DX_STAGE").alias("GFR_DX_STAGE"),
      F.max("GFR_UACR_DX_STAGE").alias("GFR_UACR_DX_STAGE"),
      F.max("UACR_STAGE").alias("UACR_STAGE"),
      F.collect_set("TREATMENT_CATEGORY").alias("CATEGORY"),
      F.collect_set("CATEGORY").alias("BRAND"))
)

pat_gfr_dx_stage_and_treatment_category = pat_gfr_dx_stage_and_treatment_category.withColumn("PULL_DT", F.current_date())

pat_gfr_dx_stage_and_treatment_category.filter(col("PATIENT_ID") == 29370266).display()

In [0]:
pat_uacr_stage_and_treatment_category = (
  df_spine_pat_journey_with_flag2
  # DATE is already at cohort-month grain from upstream notebooks
  .withColumn("SVC_DT_M", col("DATE"))
  .withColumn("TREATMENT_CATEGORY", F.explode_outer(col("TREATMENT_CATEGORY")))
  .withColumn("CATEGORY", F.explode_outer(col("CATEGORY")))
  .groupBy("PATIENT_ID", "SVC_DT_M")
  .agg(F.max("GFR_DX_STAGE").alias("GFR_DX_STAGE"),
      F.max("GFR_UACR_DX_STAGE").alias("GFR_UACR_DX_STAGE"),
      F.max("UACR_STAGE").alias("UACR_STAGE"),
      F.collect_set("TREATMENT_CATEGORY").alias("CATEGORY"),
      F.collect_set("CATEGORY").alias("BRAND"))
  .withColumn("CATEGORY", F.explode_outer(col("CATEGORY")))
  .withColumn("BRAND", F.explode_outer(col("BRAND")))
  .withColumn('CATEGORY', F.regexp_replace('CATEGORY', r'[\s]*$', ''))
)

pat_uacr_stage_and_treatment_category = pat_uacr_stage_and_treatment_category.withColumn("PULL_DT", F.current_date())

pat_uacr_stage_and_treatment_category.filter(col("PATIENT_ID") == 29370266).display()

In [0]:
pat_uacr_stage_and_treatment_category.select('UACR_STAGE').distinct().display()

In [0]:
pat_sglt2_flag = (
  df_spine_pat_journey_with_flag2
  .withColumn("TREATMENT_CATEGORY", F.explode_outer(col("TREATMENT_CATEGORY")))
  .filter("TREATMENT_CATEGORY = 'SGLT2'")
  .withColumn("SGLT2_FLAG", F.lit(1))
  .select("PATIENT_ID", "SGLT2_FLAG").distinct()
)



pat_sglt2_filter = (
  df_spine_pat_journey_with_flag2
  .withColumn("STAGE_ELIGIBLE", 
              F.when((col("GFR_UACR_DX_STAGE").isNull())|(col("GFR_UACR_DX_STAGE") == 0), -1).otherwise(
                F.when((col("GFR_UACR_DX_STAGE") == 1)|(col("GFR_UACR_DX_STAGE") == 5), 0).otherwise(1)))
  .join(pat_sglt2_flag, how = "left", on = "PATIENT_ID")
  .na.fill(value = 0, subset = "SGLT2_FLAG")
  .withColumn("ELIGIBLE_PATIENT_FLAG", 
              F.when((col("STAGE_ELIGIBLE") == 0), 0).otherwise(
                #F.when((col("GFR_UACR_DX_STAGE") == 1) & (col("DIALYSIS_FLAG") == 1), 0).otherwise(
                F.when((col("DIALYSIS_FLAG") == 1), 0).otherwise(
                F.when((col("GFR_UACR_DX_STAGE") == -1) & (col("SGLT2_FLAG") == 0), 0).otherwise(1))))
)



pat_sglt2_filter_m = (
  pat_sglt2_filter
  # DATE is already at cohort-month grain from upstream notebooks
  .withColumn("SVC_DT_M", col("DATE"))
  .groupBy("PATIENT_ID", "SVC_DT_M")
  .agg(F.max("ELIGIBLE_PATIENT_FLAG").alias("ELIGIBLE_PATIENT_FLAG"),
      F.max("GFR_UACR_DX_STAGE").alias("GFR_UACR_DX_STAGE"),
      F.max("GFR_DX_STAGE").alias("GFR_DX_STAGE"),
      F.max("UACR_STAGE").alias("UACR_STAGE"))
  .withColumn("GFR_UACR_DX_STAGE", 
            F.when((col("GFR_UACR_DX_STAGE") == 2), "STAGE_2").otherwise(
            F.when((col("GFR_UACR_DX_STAGE") == 3), "STAGE_3").otherwise(
            F.when((col("GFR_UACR_DX_STAGE") == 4), "STAGE_4").otherwise("STAGE_UNKNOWN"))))
  .withColumn("GFR_DX_STAGE", 
            F.when((col("GFR_DX_STAGE") == 2), "STAGE_2").otherwise(
            F.when((col("GFR_DX_STAGE") == 3), "STAGE_3").otherwise(
            F.when((col("GFR_DX_STAGE") == 4), "STAGE_4").otherwise("STAGE_UNKNOWN"))))
  .withColumn("UACR_STAGE", 
            F.when((col("UACR_STAGE") == 2), "STAGE_2").otherwise(
            F.when((col("UACR_STAGE") == 3), "STAGE_3").otherwise("STAGE_UNKNOWN")))
)

pat_sglt2_filter_m = pat_sglt2_filter_m.withColumn("PULL_DT", F.current_date())

pat_sglt2_filter_m.filter(col("PATIENT_ID") == 29370266).display()

In [0]:
# VALIDATION: write to separate paths
pat_new_and_total_treatments.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_pat_new_and_total_treatments_validation")

In [0]:
pat_new_ckd_stage.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_pat_new_ckd_stage_validation")

In [0]:
pat_gfr_dx_stage_and_treatment_category.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_pat_gfr_dx_stage_validation")

In [0]:
pat_uacr_stage_and_treatment_category.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_pat_uacr_stage_validation")

In [0]:
pat_sglt2_filter_m.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_pat_sglt2_filter_m_validation")

In [0]:
count_check = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_pat_sglt2_filter_m_validation")
count_check.count()